In [1]:
import pickle
import os

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

import models.vgg as models

torch.set_printoptions(sci_mode=False)

from utils import get_device, get_dataset

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = get_device(cpu=True)

N_CONCEPT, N_CLASSES, val_loader, concept_index_to_string = get_dataset("class", return_points=False, return_orig=True, return_path=False, shuffle=True)

XtoCtoY = models.End2EndModel(num_classes=N_CLASSES, num_concepts=N_CONCEPT, imagenet_weights=False, use_sigmoid=True, model_version="11_bn", use_modulelist=False)
XtoCtoY.load_state_dict(torch.load("./../saves/converted/independent-class-level-poker-seed-452.pth"))
XtoCtoY.to(device)
XtoCtoY.eval()

print("model loaded")

/home/jack/anaconda3/envs/playing_cards/lib/python3.10/site-packages/torch/cuda/__init__.py:88: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Device: cpu
model loaded


In [3]:
def generateClassMatrix(data_loader, model, n_concepts, n_classes, device):
    model.eval()

    confusion_matrix = torch.zeros(n_classes, n_concepts, dtype=torch.int32)

    with torch.no_grad():
        for images, labels, concepts, extra in tqdm(data_loader, unit="batch"):
            images, labels, concepts = images.to(device), labels.to(device), concepts.to(device)

            outputs = model(images)
            # seperate model output into task and concept predictions
            concept_out = outputs[0]
            task_out = outputs[1]
        
            _, predicted = task_out.max(1)
            
            # Binary concept predictions (present or not present)
            concept_out = concept_out >= 0.5
            
            # The matrix shows the count of concept predicted as present for the true present concepts. 100% accuracy will show three concepts with counts higher than 1 per row
            for idy, sample in enumerate(concept_out):
                confusion_matrix[predicted[idy]] += sample
                

    return confusion_matrix

In [4]:
matrix = generateClassMatrix(val_loader, XtoCtoY, N_CONCEPT, N_CLASSES, device)

print("Confusion materix")
print("=================")
print(matrix)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 188/188 [04:20<00:00,  1.39s/batch]

Confusion materix
tensor([[501, 501,   0,   0, 501,   0,   0,   0,   0,   0,   0],
        [  0,   0, 500, 501,   0, 501,   0,   0,   0,   0,   1],
        [  0, 501, 501,   0,   0,   0,   0, 501,   0,   0,   0],
        [  0,   0,   0, 501,   0,   0,   0,   0, 501, 501,   0],
        [  0,   0,   0,   0,   0,   0, 500, 500,   0,   0, 500],
        [  0,   0,   0,   0,   0, 500,   0, 500,   0,   0, 500]],
       dtype=torch.int32)
